In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import torch
import torch.nn as nn

In [ ]:

split_data = np.load("../data/train_test_splits.npz", allow_pickle=True)

train_X = split_data["train_X"]
train_Y = split_data["train_Y"]
test_X = split_data["test_X"]
test_Y = split_data["test_Y"]

num_features = train_X.shape[1]

In [ ]:
from ncps.torch import LTC
from ncps.wirings import AutoNCP

RAND_SEED = 5904

num_inputs = num_features
num_outputs = 1
num_neurons = 23
network_sparsity = 0.4

network_wiring = AutoNCP(num_neurons, num_outputs, sparsity_level=network_sparsity, seed=RAND_SEED)
model = LTC(input_size=num_inputs,
            units=network_wiring,
            batch_first=True,
            return_sequences=False,
            ode_unfolds=25)

plt.figure(figsize=(14, 10))
network_wiring.draw_graph(draw_labels=True)

In [ ]:
BATCH_SIZE = 16
BATCH_PRINT_STEP = 10

In [ ]:
class RelapseDataset(torch.utils.data.Dataset):
    def __init__(self, X, dt, Y):
        self.featuresX = torch.tensor(X, dtype=torch.float32)
        self.time = torch.tensor(dt, dtype=torch.float32)
        self.relapseOutcome = torch.tensor(Y, dtype=torch.float32)

    def __len__(self):
        return len(self.featuresX)

    def __getitem__(self, idx):
        return self.featuresX[idx], self.time[idx], self.relapseOutcome[idx]

# Doubling features along 2nd dimension
train_features_X = np.stack([train_X, train_X], axis=1)

# Adding initial time t = ~0.0 to each timespan
train_times_T = np.stack([np.zeros_like((train_Y[:,1])) + 1e-5, train_Y[:,1]], axis=1)
train_times_T = np.expand_dims(train_times_T, axis=-1)
train_times_T = np.broadcast_to(train_times_T, (train_times_T.shape[0], train_times_T.shape[1], num_neurons))

# Labels
train_labels_Y = train_Y[:,0].reshape([-1, 1])

print(train_features_X.shape)
print(train_times_T.shape)
print(train_labels_Y.shape)

# training_data = RelapseDataset(train_X, timespans_a, train_Y[:,0])
training_data = RelapseDataset(train_features_X, train_times_T, train_labels_Y)
trainloader = torch.utils.data.DataLoader(training_data, batch_size=16, shuffle=True)

In [ ]:
torch.manual_seed(RAND_SEED)
torch.cuda.manual_seed(RAND_SEED)

if(torch.cuda.is_available()):
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device:", device)
print("Training Over", len(train_X), "follow ups")

model = model.to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(0, 100):
    model.train()
    running_loss = 0.0

    for i, data in enumerate(trainloader, 0):
        features, times, labels = data

        features = features.to(device)
        times = times.to(device)
        labels = labels.to(device)
        
        # Zero gradients
        optimizer.zero_grad()
        
        # Forward
        outputs, _ = model(input=features, timespans=times)
        sigmoid_out = torch.sigmoid(outputs)

        # Backward
        loss = criterion(sigmoid_out, labels)

        loss.backward()
        
        # Optimize
        optimizer.step()
        running_loss += loss.item()

        if i % BATCH_PRINT_STEP == (BATCH_PRINT_STEP - 1):
            curr_loss = running_loss / (i + 1)
            epochBatchLossPrint = "Epoch: {} Batch: {} Loss: {:.5f}".format(epoch + 1, i + 1, curr_loss)
            print(epochBatchLossPrint)


In [ ]:
test_index = 4

print(train_features_X[0].shape)
print(train_times_T[0].shape)

single_test_X = np.stack([test_X[test_index], test_X[test_index]], axis=0)
single_test_X = torch.tensor(single_test_X).float().unsqueeze(0)
# print(single_test_X)
print(single_test_X.shape)

single_test_t = np.stack([1e-5, test_Y[:,1][test_index]])
single_test_t = np.expand_dims(single_test_t, axis=-1)
single_test_t = np.broadcast_to(single_test_t, (single_test_t.shape[0], num_neurons))
single_test_t = torch.tensor(single_test_t).float().unsqueeze(0)

# print(single_test_t)
print(single_test_t.shape)

with torch.no_grad():
    model.eval()
    model.to("cpu")
    out, _ = model(input=single_test_X, timespans=single_test_t)
    pred = torch.sigmoid(out)

    print(pred.item())
    print(test_Y[:,0][test_index])